<a href="https://colab.research.google.com/github/Dalsontimes/Colab-unsloth-gemma4/blob/main/Gemma4_MUD_QLoRA_Colab_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemma 4 E2B MUD Unsloth Colab Notebook

Gemma 4의 작은 모델은 공식적으로 `E2B` 라는 이름을 쓴다. 이 노트북은 `unsloth/gemma-4-E2B-it` 기준의 텍스트 LoRA 학습용이며, `messages` 형식과 예전 `instruction/input/output` 형식을 모두 읽을 수 있게 정리했다.


## 1) Unsloth 설치

설치 로그를 숨기지 않는다. 설치 직후 `ModuleNotFoundError: No module named 'unsloth'` 가 나면 런타임을 다시 시작한 뒤 1번 셀부터 다시 실행한다.


In [ ]:
import importlib.util
import os
import re
import subprocess
import sys


def run(cmd):
    print('$', ' '.join(cmd))
    subprocess.check_call(cmd)


if 'COLAB_' not in ''.join(os.environ.keys()):
    run([sys.executable, '-m', 'pip', 'install', '-U', 'unsloth', 'unsloth_zoo'])
else:
    import torch

    version_match = re.match(r'[\d]+\.[\d]+', str(torch.__version__))
    torch_version = version_match.group(0) if version_match else '2.10'
    xformers = {
        '2.10': 'xformers==0.0.34',
        '2.9': 'xformers==0.0.33.post1',
        '2.8': 'xformers==0.0.32.post2',
    }.get(torch_version, 'xformers==0.0.34')

    run([sys.executable, '-m', 'pip', 'install', 'sentencepiece', 'protobuf', 'datasets==4.3.0', 'huggingface_hub>=0.34.0', 'hf_transfer'])
    run([sys.executable, '-m', 'pip', 'install', '--no-deps', 'unsloth_zoo', 'bitsandbytes', 'accelerate', xformers, 'peft', 'trl', 'triton', 'unsloth'])

run([sys.executable, '-m', 'pip', 'install', '--no-deps', 'transformers==5.5.0'])
run([sys.executable, '-m', 'pip', 'install', 'torchcodec'])
run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--upgrade', 'timm'])
run([sys.executable, '-m', 'pip', 'show', 'unsloth'])
run([sys.executable, '-m', 'pip', 'show', 'unsloth_zoo'])

print('unsloth installed:', importlib.util.find_spec('unsloth') is not None)

import torch

torch._dynamo.config.recompile_limit = 64


## 2) 선택: Hugging Face 로그인

`unsloth/gemma-4-E2B-it` 를 로컬 저장만 할 때는 대체로 로그인 없이도 충분하다. 다만 Hugging Face Hub 업로드나 gated 모델 사용 예정이면 아래 줄의 주석을 풀어 실행한다.


In [ ]:
from huggingface_hub import notebook_login

print('Hub 업로드가 필요할 때만 아래 줄을 직접 실행하세요.')
# notebook_login()


## 3) 라이브러리 로드와 환경 확인

GPU 연결과 `unsloth` 설치 여부를 먼저 확인한 뒤 라이브러리를 로드한다.


In [ ]:
import importlib.util
import os
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'Unsloth requires a GPU runtime in Colab. '
        '런타임 -> 런타임 유형 변경 -> GPU 로 바꾼 뒤 '
        '세션에 다시 연결하고 설치 셀부터 다시 실행하세요.'
    )

if importlib.util.find_spec('unsloth') is None:
    raise RuntimeError(
        'Unsloth is not installed in this session. '
        '1) Unsloth 설치 셀을 다시 실행한 뒤 이 셀을 다시 실행하세요.'
    )

import unsloth  # GPU 확인 후 Unsloth 임포트
import datasets
import transformers
import trl
from datasets import load_dataset
from transformers import TextStreamer
from trl import SFTConfig, SFTTrainer
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

print('unsloth:', getattr(unsloth, '__version__', 'unknown'))
print('transformers:', transformers.__version__)
print('trl:', trl.__version__)
print('datasets:', datasets.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))
print('Capability:', torch.cuda.get_device_capability(0))


## 4) Drive 마운트와 설정

처음부터 Google Drive 경로를 쓰도록 잡아두면 런타임이 끊겨도 dataset, LoRA 출력, GGUF 파일, checkpoint 파일이 남는다. GPU 메모리 상태는 날아가지만, 저장된 checkpoint 에서 다시 이어갈 수 있다.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/gemma-mud-colab-output'
DATA_DIR = '/content/drive/MyDrive/gemma-mud-datasets'

MODEL_NAME = 'unsloth/gemma-4-E2B-it'
MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT = False  # T4 메모리가 부족하면 True 로 바꾼다
FULL_FINETUNING = False
OUTPUT_DIR = f'{BASE_DIR}/gemma4_e2b_mud_lora_out'
GGUF_DIR = f'{BASE_DIR}/gemma4_e2b_mud_gguf'
GGUF_QUANTIZATION = 'q4_k_m'

PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
NUM_TRAIN_EPOCHS = 4
LEARNING_RATE = 8e-5
WARMUP_STEPS = 5
SEED = 3407

LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0

DEFAULT_SYSTEM_PROMPT = (
    '당신은 우주항행 텍스트 MUD의 항로 안내자이자 세계관 해설자다. '
    '답변은 한국어로 하며, 서정성과 실용성을 함께 지닌다. '
    '플레이에 도움이 되는 정보와 분위기 묘사를 함께 준다.'
)

TEST_PROMPTS = [
    'First Fire Horizon이 어떤 곳인지 설명해줘.',
    'First Fire Horizon의 분위기와 역할을 설명해줘.',
    'Helios Verge이 어떤 곳인지 설명해줘.',
    'Helios Verge의 의미와 중요성을 설명해줘.',
]

DATA_CANDIDATES = [
    f'{DATA_DIR}/combined_1000.unsloth_chatml_dedup.jsonl',
    f'{DATA_DIR}/combined_1000.unsloth_gemma4_messages_dedup.jsonl',
    f'{DATA_DIR}/combined_1000.jsonl',
    '/content/gemma4_mud_alpaca_100.jsonl',
    '/content/dataset/gemma4_mud_alpaca_100.jsonl',
    '/content/drive/MyDrive/combined_1000.unsloth_chatml_dedup.jsonl',
    '/content/drive/MyDrive/combined_1000.unsloth_gemma4_messages_dedup.jsonl',
    '/content/drive/MyDrive/combined_1000.jsonl',
    '/content/gemma-mud-colab-starter/dataset/gemma4_mud_alpaca_100.jsonl',
    '/content/drive/MyDrive/gemma-mud-colab-starter/dataset/gemma4_mud_alpaca_100.jsonl',
]

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(GGUF_DIR, exist_ok=True)

print('BASE_DIR:', BASE_DIR)
print('DATA_DIR:', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('GGUF_DIR:', GGUF_DIR)


## 5) dataset 경로 찾기

가장 먼저 발견되는 dataset 파일을 사용한다. 가장 안전한 경로는 `DATA_DIR` 아래에 dataset 을 올려두는 방식이다.


In [ ]:
def resolve_data_file(candidates):
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        'dataset 파일을 찾지 못했습니다. DATA_CANDIDATES를 확인하거나 '
        'combined_1000.unsloth_chatml_dedup.jsonl, '
        'combined_1000.unsloth_gemma4_messages_dedup.jsonl, '
        'combined_1000.jsonl 중 하나를 DATA_DIR 또는 /content/ 에 올려주세요.'
    )


DATA_FILE = resolve_data_file(DATA_CANDIDATES)
print('사용할 dataset:', DATA_FILE)


## 6) Unsloth Gemma 4 E2B 모델 로드

텍스트 LoRA 경로로 로드한다. 추론과 export 안정성을 위해 Gemma 4 chat template 를 맞춘다.


In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL_NAME,
    dtype = None,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
    full_finetuning = FULL_FINETUNING,
    # token = 'hf_...',
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = 'gemma-4',
)

print('모델 로드 완료:', MODEL_NAME)


## 7) LoRA 어댑터 부착

작은 데이터셋에서 반영이 더 잘 보이도록 `r=16`, `lora_alpha=16`, `use_gradient_checkpointing='unsloth'` 를 기본값으로 둔다.


In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = LORA_R,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = SEED,
)


## 8) dataset 로드와 Gemma 4 채팅 포맷 변환

`messages` 형식은 그대로 살리고, `instruction/input/output` 형식은 표준 `system/user/assistant` 대화로 바꾼 뒤 Gemma 4 템플릿으로 `text` 컬럼을 만든다.


In [ ]:
raw_dataset = load_dataset('json', data_files=DATA_FILE, split='train')
print(raw_dataset)
print(raw_dataset[0])


def build_user_text(example):
    instruction = (example.get('instruction') or '').strip()
    extra_input = (example.get('input') or '').strip()
    if extra_input:
        return instruction + "\n\n" + extra_input
    return instruction


def normalize_message_content(content):
    if isinstance(content, str):
        text = content.strip()
        if text:
            return text
        raise ValueError('빈 문자열 content는 허용하지 않습니다.')

    if isinstance(content, dict):
        content = [content]

    if isinstance(content, list):
        parts = []
        for item in content:
            if not isinstance(item, dict):
                raise ValueError(f'지원하지 않는 content item 형식: {type(item)}')
            item_type = item.get('type', 'text')
            if item_type != 'text':
                raise ValueError(f'텍스트 전용 학습에서는 type={item_type!r} 를 사용할 수 없습니다.')
            text = str(item.get('text', '')).strip()
            if text:
                parts.append(text)
        if parts:
            return "\n".join(parts)

    raise ValueError(f'지원하지 않는 content 형식: {type(content)}')


def format_for_gemma4(example):
    if 'messages' in example and example['messages']:
        messages = []
        for message in example['messages']:
            messages.append(
                {
                    'role': message['role'],
                    'content': normalize_message_content(message['content']),
                }
            )
    else:
        required_fields = ['instruction', 'output']
        for field in required_fields:
            if field not in example:
                raise ValueError(f'필드 누락: {field}')
        messages = [
            {
                'role': 'system',
                'content': DEFAULT_SYSTEM_PROMPT,
            },
            {
                'role': 'user',
                'content': build_user_text(example),
            },
            {
                'role': 'assistant',
                'content': str(example['output']).strip(),
            },
        ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = False,
    )
    return {'text': text}


train_dataset = raw_dataset.map(
    format_for_gemma4,
    remove_columns = raw_dataset.column_names,
)

inference_system_prompt = DEFAULT_SYSTEM_PROMPT
if 'messages' in raw_dataset.column_names:
    first_messages = raw_dataset[0].get('messages') or []
    for message in first_messages:
        if message.get('role') == 'system':
            inference_system_prompt = normalize_message_content(message.get('content'))
            break

print('원본 컬럼:', raw_dataset.column_names)
print('추론용 system prompt:', inference_system_prompt)
print(train_dataset[0]['text'][:800])


## 9) Unsloth SFTTrainer 설정

현재 Gemma 4 템플릿 경로에서는 `assistant_only_loss = False` 로 두고 전체 대화 텍스트를 학습한다. 대신 epoch 를 조금 더 주고 learning rate 를 낮춰서 반영 강도를 맞춘다. 런타임 종료에 대비해 중간 checkpoint 도 Drive에 저장한다.


In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = None,
    args = SFTConfig(
        output_dir = OUTPUT_DIR,
        dataset_text_field = 'text',
        per_device_train_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
        warmup_steps = WARMUP_STEPS,
        num_train_epochs = NUM_TRAIN_EPOCHS,
        learning_rate = LEARNING_RATE,
        logging_steps = 5,
        save_strategy = 'steps',
        save_steps = 50,
        save_total_limit = 2,
        optim = 'adamw_8bit',
        weight_decay = 0.001,
        lr_scheduler_type = 'linear',
        seed = SEED,
        report_to = 'none',
        assistant_only_loss = False,
        packing = False,
    ),
)


## 10) 학습 시작

학습 중 loss 가 step마다 조금씩 흔들리는 건 정상이다. `nan` 이나 `inf` 가 아닌지만 보면 된다. 이미 저장된 checkpoint 가 있으면 가장 최근 checkpoint 에서 자동으로 이어간다.


In [ ]:
from glob import glob

checkpoints = sorted(glob(f'{OUTPUT_DIR}/checkpoint-*'))
if checkpoints:
    print('체크포인트에서 재개:', checkpoints[-1])
    trainer_stats = trainer.train(resume_from_checkpoint = True)
else:
    print('새 학습 시작')
    trainer_stats = trainer.train()

print(trainer_stats)


## 11) LoRA 어댑터 저장

여기서는 먼저 LoRA 어댑터와 tokenizer 를 저장한다. LM Studio 용 GGUF 는 뒤 셀에서 만든다.


In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print('저장 위치:', OUTPUT_DIR)
print('저장 파일:', os.listdir(OUTPUT_DIR)[:20])


## 12) 간단 추론 테스트

학습이 실제로 반영됐는지 보려면 데이터셋 원문과 가까운 질문을 greedy decoding 으로 먼저 확인한다. Gemma 4 processor 와 chat template 충돌을 피하려고, 추론은 먼저 문자열 프롬프트를 렌더링한 뒤 토크나이즈하는 2단계 방식으로 처리한다.


In [ ]:
FastModel.for_inference(model)

text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, 'tokenizer') else tokenizer


def render_text_prompt(system_prompt, user_prompt):
    messages = []
    if system_prompt:
        messages.append(
            {
                'role': 'system',
                'content': system_prompt,
            }
        )
    messages.append(
        {
            'role': 'user',
            'content': user_prompt,
        }
    )
    return text_tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = True,
    )


for prompt_text in TEST_PROMPTS:
    print('=' * 100)
    print('USER:', prompt_text)

    prompt = render_text_prompt(inference_system_prompt, prompt_text)
    inputs = tokenizer(
        text = prompt,
        return_tensors = 'pt',
        add_special_tokens = False,
    ).to('cuda')

    prompt_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 180,
            do_sample = False,
            use_cache = True,
        )

    answer = text_tokenizer.decode(
        outputs[0][prompt_len:],
        skip_special_tokens = True,
    ).split('<turn|>')[0].strip()

    print('ASSISTANT:', answer)
    print()


## 13) GGUF 생성

LM Studio 용으로는 `.gguf` 파일만 있으면 된다. 먼저 `q4_k_m` 하나를 만들어보는 흐름으로 둔다.


In [ ]:
from pathlib import Path

GGUF_DIR_PATH = Path(GGUF_DIR)
GGUF_DIR_PATH.mkdir(parents = True, exist_ok = True)

model.save_pretrained_gguf(
    str(GGUF_DIR_PATH),
    tokenizer,
    quantization_method = GGUF_QUANTIZATION,
    maximum_memory_usage = 0.5,
)

gguf_files = sorted(GGUF_DIR_PATH.glob('*.gguf'))
if not gguf_files:
    raise RuntimeError('GGUF 파일이 생성되지 않았습니다.')

print('생성된 GGUF:')
for path in gguf_files:
    print('-', path, f'({path.stat().st_size / 1024**3:.2f} GB)')


## 14) Drive 경로 확인 + 브라우저 다운로드

이 노트북은 처음부터 Drive에 저장하므로, 여기서는 생성된 GGUF 경로를 확인하고 필요할 때만 브라우저 다운로드를 건다.


In [ ]:
from google.colab import files

print('Drive 저장 경로:')
for path in gguf_files:
    print('-', path)

for path in gguf_files:
    print('다운로드 시작:', path.name)
    files.download(str(path))


## 15) 자주 흔들리는 지점

- 설치 직후 import 오류가 나면 런타임을 재시작한 뒤 다시 실행
- dataset 파일은 가능하면 `DATA_DIR` 에 올리고, 출력은 기본값 그대로 Drive 경로를 유지
- VRAM 이 부족하면 `LOAD_IN_4BIT = True` 또는 `MAX_SEQ_LENGTH = 512` 로 조정
- `messages` dataset 으로 학습했다면 추론 때도 system prompt 를 같이 넣는 편이 더 안정적
- Gemma 4는 chat template 일치가 중요하므로, 추론 때도 Gemma 4 템플릿을 그대로 사용
- GGUF 변환 중 메모리 부족이면 `maximum_memory_usage` 를 더 낮춘다
- 런타임이 끊겨도 Drive 안의 checkpoint 는 남으므로 다시 연결한 뒤 학습 셀을 다시 실행하면 이어갈 수 있다
- 작은 LoRA는 base 모델 성향을 완전히 지우는 게 아니라 덧입히는 방식이라, 테스트도 데이터셋 원문과 가까운 질문부터 보는 게 좋다
